# Moshi Family Test

| 무엇 | 결과 |
|---|---|
| 스트리밍 세션 (400ms 청크 12개, 무음 입력) | `. Oh, hello. Are you excited for` — 상태 되먹임이 이어짐 |
| 스트리밍 파형 vs 통짜 디코드 | 최대오차/RMS = 0.0044 (목표 1.0 이하) |
| 청크 생성 == 통짜 생성 (greedy) | 1·5·10프레임 청크 모두 **오디오 코드 100% 일치, 텍스트 동일** |
| 청크 인코딩 == 통짜 인코딩 | 코드 100% 일치 |

> 오차 0.0044는 A100의 cuDNN TF32(가수 10비트) 때문입니다. 끄면 1e-5 수준으로 떨어집니다.

## 1. Imports

In [ ]:
import torch
import numpy as np

from IPython.display import Audio

from project_amnesty.models import MoshiConfig, MoshiProcessor, MoshiForConditionalGeneration


device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" else torch.float32

## 2. Load Model

`MoshiProcessor`는 feature extractor / tokenizer / **Mimi 코덱**을 한 번에 묶습니다.
일반 오디오 모델과 달리 `input_values`를 내지 않고, 스트림별로 **이미 인코딩된 코드**를 냅니다:

- `audio=` → `user_audio_codes` (사용자 발화)
- `assistant_audio=` → `assistant_audio_codes` (Moshi 자신의 발화, 프롬프트용)
- `text=` → `input_ids` (오디오 프레임 수에 맞춰 자동 pad)

In [ ]:
# Set model id
model_id = "kmhf/hf-moshiko"

In [ ]:
# Load processor
processor = MoshiProcessor.from_pretrained(model_id)
processor.audio_tokenizer.to(device)

# Load model
model = MoshiForConditionalGeneration.from_pretrained(model_id, dtype=dtype)
model.to(device)

In [ ]:
# Aliasing
mimi = processor.audio_tokenizer
sr = mimi.config.sampling_rate
frame_rate = mimi.config.frame_rate

print(f"device={device} dtype={dtype} sr={sr} frame_rate={frame_rate}")

In [ ]:
# 텍스트는 오디오 프레임 수에 맞춰 자동으로 오른쪽 pad 됩니다
# (텍스트가 더 길면 `ValueError` — 오디오를 늘리거나 텍스트를 줄여야 함).

# 실제 파일: librosa.load("user.wav", sr=sr, mono=True)[0]
seconds = 5.0
user_audio = np.zeros(int(sr * seconds), dtype=np.float32)

inputs = processor(text="Hello", audio=user_audio, sampling_rate=sr)

for k, v in inputs.items():
    print(f"{k:24s} {tuple(v.shape)}")

## 3. 생성

프로세서 출력 키(`input_ids`, `user_audio_codes`, `assistant_audio_codes`)가
`generate`의 인자명과 그대로 맞아서 `**inputs`로 바로 넘어갑니다.

In [ ]:
torch.manual_seed(0)  # 샘플링 재현성

MAX_NEW_TOKENS = 64

inputs = inputs.to(device)

silence = processor.get_silence_audio_codes(MAX_NEW_TOKENS + 1, batch_size=1).to(device)
inputs["user_audio_codes"] = torch.cat([inputs["user_audio_codes"], silence], dim=2)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=True,
        temperature=0.8,
        top_k=250,
        return_audio_codes=True,
    )

text_out = processor.tokenizer.decode(output.sequences[0], skip_special_tokens=True)
print("text:", text_out)
print("audio_codes:", tuple(output.audio_codes.shape))

In [ ]:
# audio_codes 는 sequences 와 마찬가지로 프롬프트 구간을 포함한다. 여기서는 assistant_audio 를
#주지 않았으므로 그 구간은 프로세서가 채운 무음 — 즉 "사용자가 말하는 5초 동안 Moshi 는
# 침묵했다"가 그대로 돌아온 것이다. 재생할 것은 그 뒤, 실제로 생성된 프레임이다.
prompt_frames = inputs["input_ids"].shape[-1]
generated_codes = output.audio_codes[..., prompt_frames:]

with torch.no_grad():
    waveform = processor.decode_audio(generated_codes)

print(f"전체 {output.audio_codes.shape[-1]} 프레임 중 프롬프트 {prompt_frames} + 생성 {generated_codes.shape[-1]}")
print(tuple(waveform.shape), round(waveform.shape[-1] / sr, 2), "sec (생성분만)")

Audio(waveform[0, 0].float().cpu().numpy(), rate=sr)


## 4. 라이브 사용자 없이 생성 — `get_silence_audio_codes`

Moshi는 full-duplex라 매 스텝 user 프레임을 소비합니다. 사용자가 없으면 무음을 넣어야 하는데,
Mimi는 스트리밍 코덱이라 **무음이 상수 코드로 양자화되지 않습니다** (컨볼루션 상태가 프레임을 넘어 이어짐).
한 프레임을 반복하면 안 되고, 구간 전체를 한 번에 인코딩해야 해서 이 헬퍼가 있습니다.

In [ ]:
torch.manual_seed(0)  # 샘플링 재현성

NEW_TOKENS = 32

# 사용자 스트림의 길이가 곧 프롬프트 길이가 된다. 라이브 사용자가 없는 생성이라면 프롬프트는
# 한 프레임이면 충분하고, 나머지는 생성 구간을 덮을 무음이다.
# Moshi는 매 스텝 사용자 프레임을 하나씩 소비하므로 지평선 전체를 덮어야 한다.
silence_codes = processor.get_silence_audio_codes(num_frames=1 + NEW_TOKENS + 1, batch_size=1).to(device)
print("silence:", tuple(silence_codes.shape))  # (B, num_codebooks, num_frames)

# 한 프레임 반복 != 실제 무음 인코딩 (Mimi는 스트리밍 코덱이라 프레임마다 코드가 다르다)
print("frame 0 == frame 10?", torch.equal(silence_codes[..., 0], silence_codes[..., 10]))

# 프롬프트는 한 프레임. 텍스트/assistant 스트림은 "아직 말한 적 없음"으로 채워진다.
uncond = model.get_unconditional_inputs(num_samples=1)
with torch.no_grad():
    out = model.generate(
        input_ids=uncond.input_ids.to(device),
        assistant_audio_codes=uncond.assistant_audio_codes.to(device),
        user_audio_codes=silence_codes,
        max_new_tokens=NEW_TOKENS,
        do_sample=True,
    )
print(processor.tokenizer.decode(out.sequences[0], skip_special_tokens=True))


## 5. 양쪽 스트림 프롬프트

`assistant_audio`로 "Moshi가 이미 말한 내용"을 함께 주면 대화 중간부터 이어갈 수 있습니다.

In [ ]:
assistant_audio = np.zeros(int(sr * seconds), dtype=np.float32)  # 실제로는 Moshi의 이전 발화

duplex = processor(
    text="Hi there",
    audio=user_audio,
    assistant_audio=assistant_audio,
    sampling_rate=sr,
).to(device)

print(sorted(duplex.keys()))
print({k: tuple(v.shape) for k, v in duplex.items()})

## 6. 스트리밍 대화 (chunk-by-chunk full-duplex)

`generate`와 Mimi `decode` 둘 다 상태를 돌려주고 되받습니다. 그걸 이어주면 청크로 쪼갠
대화가 통짜와 프레임 단위로 같습니다.

| 상태 | 무엇 | 없으면 |
|---|---|---|
| `padding_cache` (encode) | Mimi causal conv의 프레임 간 패딩 | 청크 경계마다 클릭 노이즈 |
| `encoder_past_key_values` | Mimi encoder transformer KV | 코덱 인코더가 시간축 문맥을 잃음 |
| `streaming_state` | Moshi KV + 딜레이 마스크 + 양쪽 히스토리 + 텍스트 | 매 청크 전체 재계산이고 결과도 통짜와 달라짐 |
| `padding_cache` (decode) | 전치 conv 겹침 + 디코더 conv 패딩 + 디코더 KV | 파형이 통짜 대비 몇 배로 어긋남 |

**호출 규약**

```python
out  = model.generate(**prompt, max_new_tokens=n, concat_unconditional_inputs=True)   # 첫 청크
out  = model.generate(user_audio_codes=new, max_new_tokens=n, streaming_state=out.streaming_state)
wav  = mimi.decode(new_codes, decoder_past_key_values=..., padding_cache=..., use_streaming=True)
```

상태가 대화를 들고 있으므로 매 청크 **새로 도착한 것만** 넘기고, **이번 청크분만** 돌려받습니다.


In [ ]:
class MoshiStreamingSession:
    """청크 단위 full-duplex 세션. 사용자 오디오를 밀어넣으면 Moshi 오디오가 나온다."""

    def __init__(self, model, processor, num_codebooks=8):
        self.model = model
        self.processor = processor
        self.mimi = processor.audio_tokenizer
        self.num_codebooks = num_codebooks
        # "오디오 없음" placeholder 코드. Mimi 코드북은 0..audio_vocab_size-1 뿐이다.
        self.audio_pad = model.config.audio_vocab_size
        self.reset()

    def reset(self):
        self.encode_cache = None       # Mimi encoder: conv 패딩
        self.encoder_pkv = None        # Mimi encoder: transformer KV
        self.state = None              # Moshi: KV + 마스크 + 양쪽 히스토리 + 텍스트
        self.decode_cache = None       # Mimi decoder: 전치 conv 겹침 + conv 패딩
        self.decoder_pkv = None        # Mimi decoder: transformer KV
        self.emitted_tokens = 0        # 이미 내보낸 텍스트 토큰 수
        self.pending = None            # delay 때문에 아직 완성되지 않은 프레임

    @torch.no_grad()
    def push(self, chunk: np.ndarray, **gen_kwargs):
        """chunk: (samples,) float32 @ 24kHz. 반환: (waveform, text)"""
        wav = torch.as_tensor(chunk, dtype=torch.float32, device=self.mimi.device).reshape(1, 1, -1)

        # 1) 사용자 오디오 -> 코드 (스트리밍 인코딩)
        enc = self.mimi.encode(
            wav,
            num_quantizers=self.num_codebooks,
            encoder_past_key_values=self.encoder_pkv,
            padding_cache=self.encode_cache,
            use_streaming=True,
            return_dict=True,
        )
        self.encoder_pkv, self.encode_cache = enc.encoder_past_key_values, enc.padding_cache
        n_new = enc.audio_codes.shape[-1]

        # 2) Moshi 한 청크 생성. 첫 청크만 프롬프트에서 시작하고, 이후는 상태가 대화를 들고 간다.
        if self.state is None:
            prompt = self.model.get_unconditional_inputs(num_samples=1)
            # 프롬프트 프레임 하나를 덮을 무음 + 이번 청크.
            silence = self.processor.get_silence_audio_codes(num_frames=1, batch_size=1)
            out = self.model.generate(
                input_ids=prompt.input_ids,
                assistant_audio_codes=prompt.assistant_audio_codes,
                user_audio_codes=torch.cat([silence, enc.audio_codes], dim=-1),
                max_new_tokens=n_new,               # 프레임 수만큼만 (full-duplex 정렬)
                concat_unconditional_inputs=True,   # 첫 청크에서만
                return_audio_codes=True,
                **gen_kwargs,
            )
            fresh = out.audio_codes                 # 프롬프트 프레임이 트리밍된 채로 나온다
        else:
            before = self.state.assistant_audio_codes.shape[-1]
            out = self.model.generate(
                user_audio_codes=enc.audio_codes,   # 방금 도착한 프레임만
                max_new_tokens=n_new,
                streaming_state=self.state,
                return_audio_codes=True,
                **gen_kwargs,
            )
            fresh = out.audio_codes[..., -(out.streaming_state.assistant_audio_codes.shape[-1] - before) :]
        self.state = out.streaming_state

        # 이번 청크에서 새로 생성된 텍스트만 (안 자르면 청크마다 전체가 중복된다)
        text = self.processor.tokenizer.decode(
            out.sequences[0, self.emitted_tokens :], skip_special_tokens=True
        )
        self.emitted_tokens = out.sequences.shape[-1]

        # 3) 이번 청크의 코드만 파형으로. delay pattern 때문에 뒤쪽 코드북이 아직 도착하지 않은
        # 프레임에는 자리표시자가 남고, 그 값은 Mimi 코드북 범위 밖이라 decode 하면 죽는다.
        # 완성된 프레임까지만 내보내고 나머지는 다음 청크로 미룬다 -- 순서를 지켜야 하므로
        # 첫 미완성 프레임 이후는 전부 보류한다.
        if self.pending is not None:
            fresh = torch.cat([self.pending, fresh], dim=-1)
        done = (fresh < self.audio_pad).all(dim=1).all(dim=0)
        cut = int((~done).nonzero()[0]) if (~done).any() else fresh.shape[-1]
        ready, self.pending = fresh[..., :cut], fresh[..., cut:]
        if ready.shape[-1] == 0:
            return np.zeros(0, dtype=np.float32), text

        dec = self.mimi.decode(
            ready,
            decoder_past_key_values=self.decoder_pkv,
            padding_cache=self.decode_cache,
            use_streaming=True,
        )
        self.decoder_pkv, self.decode_cache = dec.decoder_past_key_values, dec.padding_cache
        return dec.audio_values[0, 0].float().cpu().numpy(), text


### 실행 — 400ms 청크로 밀어넣기

Mimi 프레임레이트가 12.5Hz면 1프레임 = 80ms이고, 스트리밍 인코딩은 청크가 프레임의
정수배여야 합니다. 실측하면:

| 청크 크기 | 결과 |
|---|---|
| 1프레임 (1920샘플)의 정수배 | 넣은 만큼 프레임이 나온다 |
| 1.5프레임 (2880샘플) | 남는 반프레임이 **조용히 버려진다** (6프레임분 넣고 4프레임만 나옴) |
| 1프레임 미만 (960샘플) | 컨볼루션 스택에 입력이 모자라 **크래시** |

버려지는 쪽이 특히 위험합니다. 오류 없이 오디오만 사라지므로, 청크를
`samples_per_frame`의 배수로 맞추는 책임은 호출자에게 있습니다.

**이 데모가 확인하는 것:** 상태 되먹임(padding_cache / KV / 대화 히스토리)이 제대로
이어지는지입니다. 입력은 무음이라 Moshi에게 응답할 내용이 없고, 따라서 나오는 말은
무조건부 잡담입니다 — 대화 품질을 보는 셀이 아닙니다.


In [ ]:
# 베이스 체크포인트에 무음만 주면 모델은 이어갈 문맥이 없어 서술문 중간부터 말하기도 한다.
# 이 시드는 대화로 시작하는 표본이다 (0 이면 'as it was still a popular place…' 처럼 나온다).
torch.manual_seed(7)

samples_per_frame = int(sr / frame_rate)      # 24000 / 12.5 = 1920
frames_per_chunk = 5                          # 5프레임 = 400ms
chunk_samples = samples_per_frame * frames_per_chunk
print(f"{samples_per_frame} samples/frame, chunk = {chunk_samples} samples")

session = MoshiStreamingSession(model, processor)

# 실제로는 마이크 스트림. 여기서는 5초 무음이라 Moshi 는 응답할 대상이 없고,
# 나오는 말은 무조건부 잡담이다 (이 셀이 보는 것은 상태 되먹임이 이어지는지).
stream = np.zeros(int(sr * 5.0), dtype=np.float32)
n_chunks = len(stream) // chunk_samples

audio_out, text_out = [], []
for i in range(n_chunks):
    chunk = stream[i * chunk_samples : (i + 1) * chunk_samples]
    wav, text = session.push(chunk, do_sample=True, temperature=0.8, top_k=250)
    audio_out.append(wav)
    if text.strip():
        text_out.append(text)
    print(f"chunk {i:2d}/{n_chunks}  out={wav.shape[0]:5d} samples  text={text!r}")

full = np.concatenate(audio_out)
print("\ntotal:", full.shape, full.shape[0] / sr, "sec")
print("transcript(무조건부):", " ".join(text_out))

In [ ]:
# 스트리밍으로 이어붙인 파형이 통짜 디코딩과 일치하는지 = 발음이 온전한지.
#
# 이제 세션은 청크마다 그 청크분만 디코드한다. Mimi decoder 가 전치 컨볼루션의 겹침과
# 컨볼루션 패딩을 이월해주므로, 청크 단위 디코드가 통짜 디코드와 같아야 한다.
# (한때는 decode 에 스트리밍이 없어 매번 누적 전체를 디코드하고 꼬리만 잘라 썼다 -- O(n^2).)
with torch.no_grad():
    reference = session.mimi.decode(session.state.assistant_audio_codes[..., -len(full) // samples_per_frame :])
    reference = reference.audio_values[0, 0].float().cpu().numpy()

n = min(len(full), len(reference))
err = np.abs(full[:n] - reference[:n])
rms = np.sqrt(np.mean(reference[:n] ** 2))
print(f"스트리밍 {len(full)} 샘플 vs 통짜 {len(reference)} 샘플")
print(f"최대 오차 {err.max():.5f} / 신호 RMS {rms:.5f} = {err.max() / rms:.4f}  (1.0 이하가 목표)")


### 청크 == 통짜인지 확인

이게 `streaming_state`가 실제로 하는 일입니다. 같은 사용자 오디오를 (a) 한 번에,
(b) 청크로 나눠 넣고 코드가 일치하는지 봅니다. greedy로 비교해야 의미가 있습니다
(샘플링은 청크마다 RNG 소비가 달라 애초에 비교 대상이 아닙니다).

`streaming_state`를 넘기지 않으면 여기서 어긋납니다.


In [ ]:
N_FRAMES = 30
probe = np.random.RandomState(1).randn(samples_per_frame * (N_FRAMES + 5)).astype(np.float32) * 0.05

with torch.no_grad():
    probe_codes = mimi.encode(
        torch.as_tensor(probe, device=mimi.device).reshape(1, 1, -1), num_quantizers=8
    ).audio_codes

uncond = model.get_unconditional_inputs(num_samples=1)
silence2 = processor.get_silence_audio_codes(num_frames=2, batch_size=1)
full_user = torch.cat([silence2, probe_codes[..., : N_FRAMES + 5]], dim=-1)
greedy = dict(do_sample=False, depth_decoder_do_sample=False, return_audio_codes=True)

with torch.no_grad():
    whole = model.generate(
        input_ids=uncond.input_ids,
        assistant_audio_codes=uncond.assistant_audio_codes,
        user_audio_codes=full_user,
        max_new_tokens=N_FRAMES, concat_unconditional_inputs=True, **greedy,
    )

for chunk_frames in (1, 5, 10):
    state, out, done = None, None, 0
    with torch.no_grad():
        while done < N_FRAMES:
            n = min(chunk_frames, N_FRAMES - done)
            first, done = done, done + n
            if state is None:
                out = model.generate(
                    input_ids=uncond.input_ids,
                    assistant_audio_codes=uncond.assistant_audio_codes,
                    user_audio_codes=full_user[..., : done + 5],
                    max_new_tokens=n, concat_unconditional_inputs=True, **greedy,
                )
            else:
                # 상태가 대화를 들고 있으므로 새로 도착한 프레임만 넘긴다.
                out = model.generate(
                    user_audio_codes=full_user[..., first + 5 : done + 5],
                    max_new_tokens=n, streaming_state=state, **greedy,
                )
            state = out.streaming_state
    same = (whole.audio_codes == out.audio_codes).float().mean().item()
    text_same = torch.equal(whole.sequences, out.sequences)
    print(f"{chunk_frames:2d}프레임 청크: 오디오 코드 {same:.1%} 일치, 텍스트 {'동일' if text_same else '다름'}")


In [ ]:
Audio(full, rate=sr)

### 스트리밍이 제대로 붙었는지 확인

청크로 나눠 인코딩한 결과가 통짜 인코딩과 (거의) 같아야 합니다.
`padding_cache`를 되먹이지 않으면 여기서 어긋납니다.

In [ ]:
probe = np.random.RandomState(0).randn(chunk_samples * 4).astype(np.float32) * 0.1
probe_t = torch.as_tensor(probe, device=mimi.device).reshape(1, 1, -1)

with torch.no_grad():
    whole = mimi.encode(probe_t, num_quantizers=8).audio_codes

    pc, pkv, parts = None, None, []
    for i in range(4):
        seg = probe_t[..., i * chunk_samples : (i + 1) * chunk_samples]
        e = mimi.encode(
            seg, num_quantizers=8, encoder_past_key_values=pkv,
            padding_cache=pc, use_streaming=True, return_dict=True,
        )
        pkv, pc = e.encoder_past_key_values, e.padding_cache
        parts.append(e.audio_codes)
    chunked = torch.cat(parts, dim=-1)

print("whole  :", tuple(whole.shape))
print("chunked:", tuple(chunked.shape))
n = min(whole.shape[-1], chunked.shape[-1])
agree = (whole[..., :n] == chunked[..., :n]).float().mean().item()
print(f"code agreement: {agree:.1%}")

## 7. 학습 경로 (KD용)

`forward`는 `text_labels` / `audio_labels`를 받습니다. Depth Transformer 단독 클래스도 노출돼 있어
텍스트 스트림과 오디오 스트림 로짓을 분리해 distill할 수 있습니다.

In [ ]:
import inspect

from transformers import MoshiDepthDecoderForCausalLM  # noqa: F401

print(inspect.signature(MoshiForConditionalGeneration.forward))

# 학습 경로는 세 스트림이 정확히 같은 길이여야 한다 (generate 와 달리 사용자 스트림을 미리
# 더 받지 않는다). 위에서 만든 duplex 입력이 그 조건을 만족한다.
with torch.no_grad():
    out = model(
        **duplex,
        text_labels=duplex["input_ids"],
        audio_labels=duplex["assistant_audio_codes"],
    )
print("loss:", out.loss)
